In [ ]:
#export 

import youtube_manager
import ollama_manager
import mail_manager

import logs
import logging 

import datetime
import os 
import json


tag='CENDILU'
"""
Este módulo sirve para iniciar el proceso de centinela digital para Lucía, verificando 
que los compañeros de Lucía no suben información de ella a sus canales digitales.


"""

In [2]:
#export 

def read_params(ruta_fichero: str) -> dict:
    """
    Lee un fichero de parámetros donde cada línea es un JSON con los campos:
    id, valor, descripción.
    Si el fichero no existe, crea uno con un único elemento:
    id='ultima_ejecucion', valor=<fecha actual + 7 días>, descripcion='Fecha de última ejecución'
    Devuelve un diccionario: { id: [valor, descripción] }
    """
    parametros = {}

    # Si el fichero no existe, lo creamos con un valor inicial
    if not os.path.exists(ruta_fichero):
        fecha_futura = (datetime.datetime.now() + datetime.timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%SZ")
        parametros["ultima_ejecucion"] = [fecha_futura, "Fecha de última ejecución"]
        save_params(ruta_fichero, parametros)
        return parametros

    # Si existe, lo leemos línea a línea
    with open(ruta_fichero, 'r', encoding='utf-8') as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            try:
                data = json.loads(linea)
                pid = data.get("id")
                valor = data.get("valor")
                descripcion = data.get("descripcion")
                if pid is not None:
                    parametros[pid] = [valor, descripcion]
            except json.JSONDecodeError as e:
                print(f"⚠️ Línea no válida: {linea}\nError: {e}")

    return parametros

def update_param(parametros: dict, pid: str, nuevo_valor, nueva_descripcion: str = None):
    """
    Actualiza el valor (y opcionalmente la descripción) de un parámetro existente en el diccionario.
    
    Si el id no existe, no realiza ningún cambio.
    """
    if pid in parametros:
        parametros[pid][0] = nuevo_valor
        if nueva_descripcion is not None:
            parametros[pid][1] = nueva_descripcion
    else:
        print(f"⚠️ El parámetro '{pid}' no existe. No se realizó ninguna actualización.")

def save_params(ruta_fichero: str, parametros: dict):
    """
    Escribe el diccionario de parámetros en el fichero.
    Cada línea es un JSON con los campos id, valor, descripción.
    """
    with open(ruta_fichero, 'w', encoding='utf-8') as f:
        for pid, (valor, descripcion) in parametros.items():
            data = {
                "id": pid,
                "valor": valor,
                "descripcion": descripcion
            }
            f.write(json.dumps(data, ensure_ascii=False) + '\n')

In [3]:
#expport 

def get_lista_canales_vigilar(log = logging.INFO):
    """
    La función `get_lista_canales_vigilar` devuelve una lista de canales digitales 
    que se utilizarán para vigilar la información de Lucía en el canal de sus compañeros.

    Parameters:
        None

    Exception:
        None.
    
    Returns:
        list: Una lista de canales digitales para vigilar la información de Lucía.
    """
    method="get_lista_canales_vigilar"
    
    logger, handler = logs.crea_log(log, method)
    logger.info(f"[{tag}]-{method}-START ")

    lista=[]
    lista.append({
                        'channel_id': "UCBAOfVo-X3iRZWVzFyL_X8g",
                        'channel_desc': "Josete y Manuel nunca se rinden"
                    })

    lista.append({
                        'channel_id': "UCQD51_6uLMG9k7dgezLJieA",
                        'channel_desc': "Josete el canillas"
                    })

    lista.append({
                        'channel_id': "UC9okHaIIHdPC7lVaZwMwLEA",
                        'channel_desc': "Itzel Estrellita"
                    })

    logs.cierra_log(logger, handler)

    return lista



In [4]:
#export 

def get_resumen_url(youtube_url, log = logging.INFO, resolution="480"):
    """
    The function `get_resumen_url` from an youtube url extracts the video, makes a summarization and delete the video.  
    """
    method="get_resumen_url"

    try:
        logger, handler = logs.crea_log(log, method)  

        #Makes the download of the video. 
        
        video, video_path = youtube_manager.download_video_youtube(youtube_url, resolution=resolution)

        #Makes the summarization
        resumen = ollama_manager.describe_video(video_path, model_id="mlx-community/Qwen3.5-9B-MLX-4bit")
        logger.info(f"[{tag}]-{method}- Resumen realizado {resumen}.")

        #Deletes the file. 
        os.remove(video_path)     
    except Exception as e:
        logger.error(f"[{tag}]-{method}-ERROR procesando {youtube_url} con error {e}")
    finally:
        logs.cierra_log(logger, handler)
    
    return resumen 

In [5]:
#export 

def generate_email_body(urls):
    """
    Genera el cuerpo del correo electrónico con la información de los canales y sus videos.

    Args:
        urls (list): Lista con la información recogida. Los campos que contiene son:
            - 'video_id', que representa el id del video.
            - 'title', que representa el título del video
            - 'date', que representa la fecha de publicación del video
            - 'channel_id', que representa el id del canal
            - 'channel_desc', que representa la descripción del canal.
            - pendiente 

    Returns:
        str: El cuerpo del correo.
    """
    body = ""
    current_channel_id = None
    channel_table = []

    if urls == [] or urls is None:
        return "No se han detectado videos en los últimos días."

    cabecera_tabla="<tr><th>Date</th><th>Title</th><th>Youtube URL</th><th>Visitas</th><th>Resumen del Video</th><th>Aparece Lucía</th></tr>\n"

    for video in urls:
        if video['channel_id'] != current_channel_id and current_channel_id is not None:
            # Guardar la tabla actual y comenzar una nueva
            body += f"<p><strong>Canal: {current_channel_desc}</strong></p>\n"
            body += "<table border='1'>\n"
            body += cabecera_tabla
            for row in channel_table:
                body += f"<tr>{row}</tr>\n"
            body += "</table>\n"

            # Reiniciar la tabla actual
            current_channel_id = video['channel_id']
            current_channel_desc = video['channel_desc']
            channel_table = []

        if current_channel_id is None:
            current_channel_id = video['channel_id']
            current_channel_desc = video['channel_desc']

        # Añadir la fila a la tabla actual
        date = video['date']
        video_id=video['video_id']
        title = video['title']

        #Url del video 
        youtube_url = f"https://www.youtube.com/watch?v={video_id}"
        resumen_del_video = get_resumen_url(youtube_url)
        num_visitas = youtube_manager.get_video_visits(youtube_url)

        aparece_lucia = "Sí" if "Lucía" in title else "No"
        channel_table.append(f"<td>{date}</td><td>{title}</td><td>{youtube_url}</td><td>{num_visitas}</td><td>{resumen_del_video}</td><td>{aparece_lucia}</td>")

    # Añadir la última tabla
    body += f"<p><strong>Canal: {current_channel_desc}</strong></p>\n"
    body += "<table border='1'>\n"
    body += cabecera_tabla
    for row in channel_table:
        body += f"<tr>{row}</tr>\n"
    body += "</table>\n"

    return body

In [6]:
#export

def get_lista_urls(canales, fromDays, log =logging.INFO):
    """ 
    """
    method="get_lista_urls"
    
    try:
        logger, handler = logs.crea_log(log, method)  
        logger.info(f"[{tag}]-{method}-START Iniciando obtención todas urls a vigilar")

        urls= []
        #Procesamos todos los canales. 
        for canal in canales:
            id = canal['channel_id']
            desc = canal['channel_desc']   

            #Recuperamos la lista de videos del canal en marcha. 
            videos = youtube_manager.get_list_videos(id, fromDays=fromDays)
            logger.info(f"[{tag}]-{method}-INFO {id} - {desc} con {len(videos)} videos en {fromDays} días")

            #Añadimos el id, desc a la lista de videos. 
            for video in videos:
                video["channel_id"] = id
                video["channel_desc"] = desc

            urls.extend(videos)

        return urls
    except Exception as e:
        logger.error(f"[{tag}]-{method}-ERROR {e}")
        raise e
    finally:
        logs.cierra_log(logger, handler)


In [7]:
#export

def main_comprueba_youtube_clase(log=logging.INFO, fromDays=None, ficheroIni='cendilu.ini'):
    """ 
    """
    method="main_comprueba_youtube_clase"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    #0.- Leemos los parámetros para recuperar la última fecha de ejecución. Si nos han pedido con una fecha concreta la actualizamos.
    params = read_params(ruta_fichero=ficheroIni)
    if fromDays is not None:
        update_param(params, "ultima_ejecucion", fromDays.strftime("%Y-%m-%dT%H:%M:%SZ"))
    ts=params.get("ultima_ejecucion")[0]
    ts=datetime.datetime.strptime(ts, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=datetime.timezone.utc)

    #1.- Recuperamos la lista de canales que tenemos que vigilar
    canales = get_lista_canales_vigilar()

    #2.- Descargamos la lista de urls completas que tenemos que comprobar
    urls = get_lista_urls(canales, fromDays=ts)
    sorted_urls = sorted(urls, key=lambda x: (x['channel_id'], x['date']), reverse=True)
    
    # Actualizamos el parámetro. 
    update_param(params, 'ultima_ejecucion', datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%SZ"))

    #3.- Para cada video, transcribimos, comprobamos si habla de lucía o si ha metido la cara de lucía. 

    #4.- Enviamos un correo electrónico con el resultado de la comprobación. 
    to_email="zzddfge@gmail.com"
    body = generate_email_body(sorted_urls)
    subject=f"[cendilu] - {datetime.date.today()} - Comprobación de videos en YouTube"

    mail_manager.send_email(subject, body, to_email, log=log)
    
    to_email="mjgaray76@gmail.com"
    mail_manager.send_email(subject, body, to_email, log=log)
    
    #5.- Grabamos el parámetro en el fichero.
    save_params(ficheroIni, params)
    #print(urls)
    


In [8]:
##########################################################
##########################################################
##########################################################

In [ ]:

main_comprueba_youtube_clase()

In [ ]:
!pip install -U yt-dlp

In [ ]:
!pip install -U mlx-lm
!pip install -U mlx-vlm
!pip install -U mlx-audio

